# Superstore Customer & Revenue Intelligence — Final Python Analysis

**Purpose:** validate the business KPIs used in the Tableau dashboard and create repeatable customer, sales, profitability, and RFM analyses.

**Important fixes in this version**
- Reads an uploaded Excel (`.xls`/`.xlsx`) or CSV file instead of relying on one hard-coded filename.
- Cleans date and numeric columns consistently.
- Uses the same six rule-based RFM segments as the Tableau dashboard.
- Separates **average customer sales** from **average order value**.
- Validates the baseline values before analysis.

Expected source: the Superstore `Orders` table (9,994 rows). If you upload CSV, it should contain the same 21 columns as the `Orders` sheet.

In [ ]:
import sys, subprocess, importlib.util, pathlib, io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Excel .xls support may require xlrd in a fresh Google Colab runtime.
if importlib.util.find_spec("xlrd") is None:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "xlrd"])
    except Exception:
        print("Could not install xlrd automatically. Upload a CSV or .xlsx file instead.")

print("Libraries loaded successfully.")

## 1. Load the Superstore data

In Google Colab, upload the original Excel file or a CSV export. The code chooses the `Orders` worksheet when present.

In [ ]:
# Google Colab upload. If running locally, set FILE_PATH to your data file instead.
try:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded.")
    FILE_PATH = next(iter(uploaded.keys()))
except ImportError:
    FILE_PATH = "_superstore(2).xls"  # Change this path if running outside Colab.

path = pathlib.Path(FILE_PATH)
if not path.exists():
    raise FileNotFoundError(f"File not found: {path}. Upload the Superstore file or update FILE_PATH.")

suffix = path.suffix.lower()
if suffix in [".xlsx", ".xls"]:
    xls = pd.ExcelFile(path)
    sheet = "Orders" if "Orders" in xls.sheet_names else xls.sheet_names[0]
    df = pd.read_excel(path, sheet_name=sheet)
elif suffix == ".csv":
    # Auto-detect comma vs semicolon delimiters.
    try:
        df = pd.read_csv(path, sep=None, engine="python", encoding="latin-1")
    except Exception:
        df = pd.read_csv(path, encoding="latin-1")
else:
    raise ValueError("Use an Excel .xls/.xlsx file or a .csv file.")

# Normalize accidental spaces in column headers.
df.columns = df.columns.astype(str).str.strip()
print("Loaded file:", path.name)
print("Rows and columns:", df.shape)
print("Columns:", list(df.columns))
display(df.head())

## 2. Clean and validate the data

Dates are parsed day-first where needed. Numeric values are converted safely, including comma decimal separators when present.

In [ ]:
required = ["Row ID", "Order ID", "Order Date", "Ship Date", "Customer ID", "Customer Name", "Segment", "Region", "Category", "Sub-Category", "Sales", "Quantity", "Discount", "Profit"]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Parse dates; values already stored as datetime remain valid.
for col in ["Order Date", "Ship Date"]:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")

# Robust numeric conversion: handle numeric types, comma decimals, and common thousands separators.
def parse_number(series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    s = series.astype(str).str.strip().str.replace("$", "", regex=False).str.replace("€", "", regex=False)
    def normalize(v):
        if v in ["", "nan", "None", "NaN"]:
            return np.nan
        if "," in v and "." in v:
            # The right-most punctuation is treated as the decimal separator.
            if v.rfind(",") > v.rfind("."):
                v = v.replace(".", "").replace(",", ".")
            else:
                v = v.replace(",", "")
        elif "," in v:
            v = v.replace(",", ".")
        return v
    return pd.to_numeric(s.map(normalize), errors="coerce")

for col in ["Sales", "Quantity", "Discount", "Profit"]:
    df[col] = parse_number(df[col])

# Quality report before any row removal.
quality = pd.DataFrame({
    "metric": ["Rows", "Unique Row IDs", "Duplicate Row IDs", "Missing Order IDs", "Missing Customer IDs", "Invalid Order Dates", "Invalid Ship Dates", "Missing Sales", "Missing Profit"],
    "value": [len(df), df["Row ID"].nunique(dropna=True), int(df["Row ID"].duplicated().sum()), int(df["Order ID"].isna().sum()), int(df["Customer ID"].isna().sum()), int(df["Order Date"].isna().sum()), int(df["Ship Date"].isna().sum()), int(df["Sales"].isna().sum()), int(df["Profit"].isna().sum())]
})
display(quality)

# Do not silently drop rows. Stop if required fields are invalid so the issue can be investigated.
critical = df[["Row ID", "Order ID", "Customer ID", "Order Date", "Ship Date", "Sales", "Profit"]].isna().sum()
if critical.any():
    raise ValueError("Critical missing/invalid values found. Review the quality report before continuing:\n" + critical[critical > 0].to_string())

# Derived columns used by analysis.
df["Shipping Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Order Month"] = df["Order Date"].dt.to_period("M")
df["Profit Margin"] = np.where(df["Sales"] != 0, df["Profit"] / df["Sales"], np.nan)

print("Data cleaning complete. Rows retained:", len(df))

## 3. Core KPIs — validated baseline

Definitions:
- **Total customers:** distinct `Customer ID`.
- **Repeat customers:** customers with more than one distinct `Order ID`.
- **Top 10 customer sales share:** top 10 customers by sales divided by all sales.
- **Average customer sales/value:** total sales divided by distinct customers.
- **Average order value:** total sales divided by distinct orders.
- **Profit margin:** total profit divided by total sales.

In [ ]:
total_rows = len(df)
total_orders = df["Order ID"].nunique()
total_customers = df["Customer ID"].nunique()
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
profit_margin = total_profit / total_sales if total_sales else np.nan
average_customer_sales = total_sales / total_customers if total_customers else np.nan
average_order_value = total_sales / total_orders if total_orders else np.nan

orders_by_customer = df.groupby("Customer ID")["Order ID"].nunique()
repeat_customers = int((orders_by_customer > 1).sum())
one_time_customers = int((orders_by_customer == 1).sum())
repeat_customer_pct = repeat_customers / total_customers * 100 if total_customers else np.nan

customer_sales = df.groupby("Customer ID")["Sales"].sum().sort_values(ascending=False)
top10_sales = customer_sales.head(10).sum()
top10_sales_share = top10_sales / total_sales * 100 if total_sales else np.nan

kpis = pd.DataFrame({
    "KPI": ["Rows", "Distinct orders", "Total customers", "Repeat customers", "One-time customers", "Repeat customer rate (%)", "Total sales", "Total profit", "Profit margin (%)", "Average customer sales", "Average order value", "Top 10 customer sales share (%)"],
    "Value": [total_rows, total_orders, total_customers, repeat_customers, one_time_customers, repeat_customer_pct, total_sales, total_profit, profit_margin*100, average_customer_sales, average_order_value, top10_sales_share]
})
display(kpis.style.format({"Value": "{:,.2f}"}))

# Source-data checks for the standard Superstore Orders table.
expected = {"Rows": 9994, "Distinct orders": 5009, "Total customers": 793}
observed = {"Rows": total_rows, "Distinct orders": total_orders, "Total customers": total_customers}
for name, target in expected.items():
    if observed[name] != target:
        print(f"WARNING: {name} is {observed[name]:,}; standard Superstore baseline is {target:,}. Check the imported file and worksheet.")

print("\nValidated standard Superstore baseline (if the correct Orders data is loaded):")
print(f"Rows: {total_rows:,} | Orders: {total_orders:,} | Customers: {total_customers:,}")
print(f"Sales: {total_sales:,.2f} | Profit: {total_profit:,.2f} | Margin: {profit_margin*100:.2f}%")
print(f"Repeat customers: {repeat_customers:,} ({repeat_customer_pct:.2f}%)")
print(f"Average customer sales: {average_customer_sales:,.2f} | Average order value: {average_order_value:,.2f}")
print(f"Top 10 customer sales share: {top10_sales_share:.2f}%")

## 4. Customer-level profitability and top customers

In [ ]:
customer_summary = df.groupby(["Customer ID", "Customer Name"], as_index=False).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Orders=("Order ID", "nunique")
)
customer_summary["Profit_Margin"] = np.where(customer_summary["Total_Sales"] != 0, customer_summary["Total_Profit"] / customer_summary["Total_Sales"], np.nan)
customer_summary["Sales_Per_Order"] = customer_summary["Total_Sales"] / customer_summary["Total_Orders"]
customer_summary["Profit_Per_Order"] = customer_summary["Total_Profit"] / customer_summary["Total_Orders"]

print("Top 10 customers by profit")
display(customer_summary.nlargest(10, "Total_Profit").reset_index(drop=True))
print("Top 10 customers by sales")
display(customer_summary.nlargest(10, "Total_Sales").reset_index(drop=True))
loss_making_customers = customer_summary[customer_summary["Total_Profit"] < 0].sort_values("Total_Profit")
print("Loss-making customers:", len(loss_making_customers))
display(loss_making_customers.head(10))

## 5. Sales, profit, category, region, and discount analysis

In [ ]:
yearly_performance = df.groupby("Year", as_index=False).agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"))
yearly_performance["Profit_Margin_Pct"] = yearly_performance["Total_Profit"] / yearly_performance["Total_Sales"] * 100
yearly_performance["Sales_YoY_Growth_Pct"] = yearly_performance["Total_Sales"].pct_change() * 100
print("Yearly performance")
display(yearly_performance.round(2))

category_performance = df.groupby("Category", as_index=False).agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"))
category_performance["Profit_Margin_Pct"] = category_performance["Total_Profit"] / category_performance["Total_Sales"] * 100
print("Category performance")
display(category_performance.sort_values("Total_Profit", ascending=False).round(2))

subcategory_performance = df.groupby("Sub-Category", as_index=False).agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"), Average_Discount=("Discount", "mean"))
subcategory_performance["Profit_Margin_Pct"] = subcategory_performance["Total_Profit"] / subcategory_performance["Total_Sales"] * 100
print("Lowest-profit subcategories")
display(subcategory_performance.sort_values("Total_Profit").head(10).round(2))

region_performance = df.groupby("Region", as_index=False).agg(Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"))
region_performance["Profit_Margin_Pct"] = region_performance["Total_Profit"] / region_performance["Total_Sales"] * 100
print("Regional performance")
display(region_performance.sort_values("Total_Sales", ascending=False).round(2))

discount_performance = df.groupby("Discount", as_index=False).agg(Rows=("Row ID", "count"), Total_Sales=("Sales", "sum"), Total_Profit=("Profit", "sum"))
discount_performance["Profit_Margin_Pct"] = discount_performance["Total_Profit"] / discount_performance["Total_Sales"] * 100
print("Profitability by discount level")
display(discount_performance.sort_values("Discount").round(2))

## 6. RFM customer segmentation — aligned with Tableau

This uses the same rule-based logic as the Tableau dashboard, with the latest order date in the dataset as the reference date:
- Champions: Recency ≤ 90 days, Frequency ≥ 5 orders, Monetary sales ≥ 1,000.
- New Customers: Recency ≤ 30 days and exactly 1 order.
- Potential Loyalists: Recency ≤ 90 days and Frequency ≤ 2 orders.
- Loyal Customers: Recency ≤ 180 days and Frequency ≥ 3 orders.
- At Risk: Recency > 180 days and Frequency ≥ 3 orders.
- Remaining customers: Lost Customers.

In [ ]:
reference_date = df["Order Date"].max()
rfm = df.groupby("Customer ID", as_index=False).agg(
    Last_Order=("Order Date", "max"),
    Frequency=("Order ID", "nunique"),
    Monetary=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
)
rfm["Recency_Days"] = (reference_date - rfm["Last_Order"]).dt.days

def assign_rfm_segment(row):
    recency, frequency, monetary = row["Recency_Days"], row["Frequency"], row["Monetary"]
    if recency <= 90 and frequency >= 5 and monetary >= 1000:
        return "Champions"
    elif recency <= 30 and frequency == 1:
        return "New Customers"
    elif recency <= 90 and frequency <= 2:
        return "Potential Loyalists"
    elif recency <= 180 and frequency >= 3:
        return "Loyal Customers"
    elif recency > 180 and frequency >= 3:
        return "At Risk"
    else:
        return "Lost Customers"

rfm["RFM_Segment"] = rfm.apply(assign_rfm_segment, axis=1)
rfm_summary = rfm.groupby("RFM_Segment", as_index=False).agg(
    Customers=("Customer ID", "nunique"),
    Sales=("Monetary", "sum"),
    Profit=("Total_Profit", "sum")
)
rfm_summary["Customer_Share_Pct"] = rfm_summary["Customers"] / total_customers * 100
rfm_summary["Profit_Margin_Pct"] = np.where(rfm_summary["Sales"] != 0, rfm_summary["Profit"] / rfm_summary["Sales"] * 100, np.nan)
print("RFM segment summary")
display(rfm_summary.sort_values("Customers", ascending=False).round(2))

# Export useful tables for portfolio documentation or follow-up visualization.
rfm.to_csv("rfm_customer_segments.csv", index=False)
customer_summary.to_csv("customer_profitability.csv", index=False)
rfm_summary.to_csv("rfm_segment_summary.csv", index=False)
print("Exported rfm_customer_segments.csv, customer_profitability.csv, and rfm_segment_summary.csv")

## 7. Visual summaries

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
year_plot = yearly_performance.sort_values("Year")
axes[0].plot(year_plot["Year"], year_plot["Total_Sales"], marker="o", label="Sales")
axes[0].set_title("Sales by Year")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Sales")
axes[0].grid(True, alpha=0.3)

seg_plot = rfm_summary.sort_values("Customers", ascending=True)
axes[1].barh(seg_plot["RFM_Segment"], seg_plot["Customers"])
axes[1].set_title("Customers by RFM Segment")
axes[1].set_xlabel("Number of Customers")
axes[1].grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Final business summary

Use the tables above to write findings based on the actual results. Do not assume a region or category is the worst performer until the output confirms it.

In [ ]:
best_category = category_performance.loc[category_performance["Total_Profit"].idxmax()]
worst_category = category_performance.loc[category_performance["Total_Profit"].idxmin()]
best_region = region_performance.loc[region_performance["Total_Profit"].idxmax()]
worst_region = region_performance.loc[region_performance["Total_Profit"].idxmin()]

print("FINAL PROJECT SUMMARY")
print("=" * 60)
print(f"Rows analyzed: {total_rows:,}")
print(f"Customers: {total_customers:,} | Orders: {total_orders:,}")
print(f"Sales: {total_sales:,.2f} | Profit: {total_profit:,.2f} | Profit margin: {profit_margin*100:.2f}%")
print(f"Repeat customers: {repeat_customers:,} ({repeat_customer_pct:.2f}%)")
print(f"Top 10 customer sales share: {top10_sales_share:.2f}%")
print(f"Most profitable category: {best_category['Category']} ({best_category['Total_Profit']:,.2f} profit)")
print(f"Lowest-profit category: {worst_category['Category']} ({worst_category['Total_Profit']:,.2f} profit)")
print(f"Highest-profit region: {best_region['Region']} ({best_region['Total_Profit']:,.2f} profit)")
print(f"Lowest-profit region: {worst_region['Region']} ({worst_region['Total_Profit']:,.2f} profit)")
print("\nPython analysis complete. Cross-check totals against SQL and Tableau before publishing.")